## Project Title & Team Info

**Project Title**: _Workshop 1: WiDS University Datathon 2026_  
**Team Name**: _Research DUO_  
**University**: _Bucharest University of Economic Studies_  
**Course**: _Machine Learning_  
**Term**: _1st Semester, 2025_  

**Team Members**:  
- Leonardo-Gabriel MARCU
- Antonia-Paula MITREA
- Maria FOLEANU

In [24]:
!pip install kaggle

In [25]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"leonardomarcu","key":"90e4f9e2c12bb3de5825ba1691c7ad2d"}'}

In [26]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [27]:
!kaggle competitions download -c wids-university-datathon-2025


wids-university-datathon-2025.zip: Skipping, found more recently modified local copy (use --force to force download)


In [28]:
!unzip wids-university-datathon-2025.zip -d data/

Archive:  wids-university-datathon-2025.zip
replace data/WiDS _-_ Watch Duty_ Data Dictionary.docx? [y]es, [n]o, [A]ll, [N]one, [r]ename: Y
  inflating: data/WiDS _-_ Watch Duty_ Data Dictionary.docx  
replace data/evac_zone_status_geo_event_map.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: Y
  inflating: data/evac_zone_status_geo_event_map.csv  
replace data/evac_zones_gis_evaczone.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: Y
  inflating: data/evac_zones_gis_evaczone.csv  Y

replace data/evac_zones_gis_evaczonechangelog.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename:   inflating: data/evac_zones_gis_evaczonechangelog.csv  Y
Y
Y
Y
Y
Y

replace data/fire_perimeters_gis_fireperimeter.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename:   inflating: data/fire_perimeters_gis_fireperimeter.csv  Y

replace data/fire_perimeters_gis_fireperimeterchangelog.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename:   inflating: data/fire_perimeters_gis_fireperimeterchangelog.csv  
replace data/geo_events_externalgeoevent.csv? [y

In [29]:
from pathlib import Path

EXT_DIR = Path("data/external")
EXT_DIR.mkdir(parents=True, exist_ok=True)
print("External data folder:", EXT_DIR.resolve())


External data folder: /content/data/external


Pasul 2 — Data Discovery (înțelegerea datelor)
2.1 Scopul etapei de Data Discovery

Înainte de orice procesare sau modelare, este esențial să înțelegem:

structura fiecărui fișier (dimensiuni, coloane),

tipurile de date și lipsurile (missing values),

ce informație este statică vs. temporală,

unde se află informația critică (JSON-uri, changelog-uri),

cum se pot lega fișierele între ele.

Această etapă ne ajută să evităm erori conceptuale (ex. data leakage) și să definim corect unitatea de analiză și targetul ulterior.

In [31]:
import pandas as pd

# fișiere competiție
evac_zone_map = pd.read_csv("data/evac_zone_status_geo_event_map.csv")
evac_zones = pd.read_csv("data/evac_zones_gis_evaczone.csv")
evac_zone_changelog = pd.read_csv("data/evac_zones_gis_evaczonechangelog.csv")

fires = pd.read_csv("data/geo_events_geoevent.csv")
fire_changelog = pd.read_csv("data/geo_events_geoeventchangelog.csv")

fire_perimeters = pd.read_csv("data/fire_perimeters_gis_fireperimeter.csv")
fire_perimeters_changelog = pd.read_csv("data/fire_perimeters_gis_fireperimeterchangelog.csv")

# date externe
svi = pd.read_csv("data/external/SVI_2022_US.csv")


In [32]:
datasets = {
    "evac_zone_map": evac_zone_map,
    "evac_zones": evac_zones,
    "evac_zone_changelog": evac_zone_changelog,
    "fires": fires,
    "fire_changelog": fire_changelog,
    "fire_perimeters": fire_perimeters,
    "fire_perimeters_changelog": fire_perimeters_changelog,
    "SVI_2022": svi
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("shape:", df.shape)



evac_zone_map
shape: (4429, 3)

evac_zones
shape: (37458, 16)

evac_zone_changelog
shape: (68919, 4)

fires
shape: (62696, 17)

fire_changelog
shape: (178697, 5)

fire_perimeters
shape: (6207, 14)

fire_perimeters_changelog
shape: (9048, 5)

SVI_2022
shape: (84120, 158)


In [33]:
for name, df in datasets.items():
    print(f"\n{name} – dtypes")
    display(df.dtypes)



evac_zone_map – dtypes


,0
date_created,object
uid_v2,object
geo_event_id,int64



evac_zones – dtypes


,0
id,int64
date_created,object
date_modified,object
uid_v2,object
is_active,bool
display_name,object
region_id,int64
source_attribution,object
dataset_name,object
source_extra_data,object



evac_zone_changelog – dtypes


...

,0
id,int64
date_created,object
date_modified,object
geo_event_id,float64
approval_status,object
source,object
source_unique_id,object
source_date_current,object
source_incident_name,object
source_acres,float64



fire_perimeters_changelog – dtypes


,0
id,int64
date_created,object
user_created_id,int64
changes,object
fire_perimeter_id,int64



SVI_2022 – dtypes


,0
ST,int64
STATE,object
ST_ABBR,object
STCNTY,int64
COUNTY,object
...,...
MP_NHPI,float64
EP_TWOMORE,float64
MP_TWOMORE,float64
EP_OTHERRACE,float64


In [34]:
for name, df in datasets.items():
    print(f"\n{name} – missing values (%)")
    display((df.isna().mean() * 100).sort_values(ascending=False).head(10))



evac_zone_map – missing values (%)


,0
date_created,0.0
uid_v2,0.0
geo_event_id,0.0



evac_zones – missing values (%)


,0
pending_updates,100.000000
status,99.687650
external_status,12.648833
display_name,0.016018
uid_v2,0.000000
id,0.000000
date_modified,0.000000
date_created,0.000000
source_attribution,0.000000
region_id,0.000000



evac_zone_changelog – missing values (%)


...

,0
geo_event_id,13.017561
source_acres,10.423715
source_extra_data,2.481070
source_incident_name,0.032222
date_created,0.000000
id,0.000000
source,0.000000
approval_status,0.000000
date_modified,0.000000
source_date_current,0.000000



fire_perimeters_changelog – missing values (%)


,0
id,0.0
date_created,0.0
user_created_id,0.0
changes,0.0
fire_perimeter_id,0.0



SVI_2022 – missing values (%)


,0
ST,0.0
STATE,0.0
ST_ABBR,0.0
STCNTY,0.0
COUNTY,0.0
FIPS,0.0
LOCATION,0.0
AREA_SQMI,0.0
E_TOTPOP,0.0
M_TOTPOP,0.0


In [35]:
print("Fire IDs:", fires["id"].nunique())
print("Fire IDs in changelog:", fire_changelog["geo_event_id"].nunique())
print("Fire IDs in evac map:", evac_zone_map["geo_event_id"].nunique())


Fire IDs: 62696
Fire IDs in changelog: 42231
Fire IDs in evac map: 483


In [36]:
fires["data"].head(3)


,data
0,"{""is_fps"": false, ""acreage"": 50, ""containment""..."
1,"{""is_fps"": false, ""acreage"": 0, ""containment"":..."
2,"{""is_fps"": false, ""acreage"": 0, ""containment"":..."


In [37]:
fire_changelog["changes"].head(5)


,changes
0,"{""name"": [""Vegetation Fire"", ""Charlotte Fire""]}"
1,"{""data.links"": [[], [{""label"": ""Pulsepoint Inc..."
2,"{""address"": [""W Ave C & 110th St W, Antelope A..."
3,{}
4,"{""data.acreage"": [null, 3]}"


In [38]:
evac_zone_changelog["changes"].head(5)


,changes
0,"{""geom"": [""POLYGON ((-112.33752250671387 34.67..."
1,"{""geom"": [""POLYGON ((-112.35966682434082 34.66..."
2,"{""geom"": [""MULTIPOLYGON (((-121.706523 39.3685..."
3,"{""geom"": [""POLYGON ((-121.706784 39.364853, -1..."
4,"{""geom"": [""POLYGON ((-121.703858 39.366719, -1..."


In [43]:
svi.shape
svi.columns.tolist()[:20]

['ST',
 'STATE',
 'ST_ABBR',
 'STCNTY',
 'COUNTY',
 'FIPS',
 'LOCATION',
 'AREA_SQMI',
 'E_TOTPOP',
 'M_TOTPOP',
 'E_HU',
 'M_HU',
 'E_HH',
 'M_HH',
 'E_POV150',
 'M_POV150',
 'E_UNEMP',
 'M_UNEMP',
 'E_HBURD',
 'M_HBURD']

In [40]:
svi[["FIPS", "RPL_THEMES", "RPL_THEME1", "RPL_THEME2", "RPL_THEME3", "RPL_THEME4"]].describe()


,FIPS,RPL_THEMES,RPL_THEME1,RPL_THEME2,RPL_THEME3,RPL_THEME4
count,8.412000e+04,84120.000000,84120.000000,84120.000000,84120.000000,84120.000000
mean,2.786952e+10,-8.744074,-8.744081,-8.553981,-5.477168,-8.542111
std,1.593582e+10,95.677502,95.677502,94.697657,77.058432,94.636066
min,1.001020e+09,-999.000000,-999.000000,-999.000000,-999.000000,-999.000000
25%,1.211991e+10,0.243000,0.243000,0.243100,0.243900,0.243175
50%,2.714501e+10,0.495300,0.495300,0.495400,0.496000,0.495400
75%,4.106703e+10,0.747700,0.747625,0.747700,0.748300,0.747700
max,5.604595e+10,1.000000,1.000000,1.000000,0.995400,1.000000


## Interpretare generală – Data Discovery

Analiza exploratorie inițială a confirmat că datele puse la dispoziție pentru Track 1 au o structură complexă, profund temporală, fiind distribuite în mai multe tabele specializate care surprind perspective diferite asupra aceluiași fenomen: evoluția incendiilor și reacția instituțională prin ordine de evacuare.

Structura și amploarea datelor

Setul de date conține 62.696 evenimente de tip incendiu, însă doar o mică parte dintre acestea (483) sunt asociate explicit cu zone de evacuare prin tabelul de mapare. Această observație este importantă din punct de vedere conceptual: majoritatea incendiilor nu generează evacuări, iar analiza întârzierilor este relevantă doar pentru un subset restrâns, dar critic, de evenimente cu risc ridicat.

Changelog-urile (atât pentru incendii, cât și pentru zonele de evacuare) sunt considerabil mai mari decât tabelele de bază, ceea ce confirmă că informația cheie este de natură temporală, iar înțelegerea succesiunii evenimentelor este esențială pentru definirea corectă a întârzierilor.

Calitatea și natura variabilelor

Majoritatea variabilelor esențiale (identificatori, timestamp-uri, legături între tabele) sunt complet populate, ceea ce indică o bună integritate structurală a datelor. Valorile lipsă apar în principal în:

câmpuri descriptive (description, address), care nu sunt critice pentru obiectivul proiectului;

câmpuri administrative (incident_id, external_id), care pot fi ignorate fără a afecta analiza întârzierilor;

tabelele de perimetre, unde lipsa unor legături către evenimentele de incendiu sugerează că nu toate incendiile beneficiază de actualizări geografice oficiale.

Un aspect important identificat este faptul că unele identificatoare (geo_event_id, id) apar ca valori de tip float, ca urmare a prezenței unor valori lipsă. Aceasta necesită o standardizare explicită în etapa de procesare pentru a evita erori subtile la îmbinarea tabelelor.

Rolul câmpurilor JSON și al changelog-urilor

Câmpurile data și changes, stocate sub formă de JSON, reprezintă nucleul informațional al dataset-ului. Din explorarea preliminară reiese că:

geo_events_geoevent.data conține indicatori operaționali precum acreage, containment și is_fps, care descriu starea curentă a incendiului;

geo_events_geoeventchangelog.changes surprinde semnale timpurii de risc (ex. intensificarea propagării, amenințări asupra structurilor), fiind o sursă esențială pentru definirea momentului în care un incendiu devine periculos;

evac_zones_gis_evaczonechangelog.changes documentează schimbările oficiale de status ale zonelor de evacuare (warning, order), dar include și numeroase modificări geometrice, care trebuie filtrate în etapele următoare.

Această structură confirmă necesitatea unei procesări dedicate a câmpurilor JSON și a unei selecții explicite a tipurilor de schimbări relevante pentru obiectivul analizei.

Integrarea datelor de vulnerabilitate (SVI 2022)

Setul de date extern SVI 2022 oferă o acoperire completă la nivel de census tract (84.120 unități) și include atât scoruri agregate de vulnerabilitate (RPL_THEMES), cât și variabile tematice detaliate (socioeconomic, household composition, minority status, housing & transportation).

O observație critică este prezența valorilor -999 în variabilele de tip percentile (RPL_*), care indică lipsa scorului pentru anumite unități geografice. Aceste valori trebuie tratate explicit ca valori lipsă înainte de orice analiză statistică sau modelare, pentru a evita distorsionarea concluziilor legate de echitate.

Implicații pentru pașii următori

Rezultatele Data Discovery sugerează că:

unitatea naturală de analiză va fi evenimentul de incendiu, cu o componentă temporală explicită;

măsurarea întârzierilor va fi posibilă doar pentru incendiile asociate cu zone de evacuare, în timp ce restul pot fi utilizate ca observații negative sau cenzurate în modelarea predictivă;

procesarea corectă a timpului, a identificatorilor și a câmpurilor JSON este o condiție necesară pentru definirea unui target valid;

datele de vulnerabilitate oferă un cadru solid pentru analiza disparităților și pentru evaluarea impactului soluției asupra comunităților expuse.

În ansamblu, datele sunt suficient de bogate și coerente pentru a susține atât o analiză riguroasă a întârzierilor, cât și dezvoltarea unui model predictiv cu relevanță operațională, cu condiția unei etape de procesare atent controlate.

## Pasul 3 — Data Processing
**3.0 Scopul etapei de Data Processing**

După etapa de Data Discovery, este necesară o procesare riguroasă a datelor pentru a asigura coerența temporală, corectitudinea legăturilor dintre tabele și extragerea informației relevante pentru analiza întârzierilor și pentru modelarea predictivă.

În această etapă urmărim:

standardizarea formatelor de timp și a identificatorilor,

curățarea valorilor lipsă sau codificate special,

parsarea câmpurilor JSON care conțin informația critică,

pregătirea dataset-urilor pentru definirea unității de analiză și a variabilei țintă.

Această etapă este esențială pentru a evita erori conceptuale (ex. join-uri incorecte, data leakage) și pentru a construi o bază solidă pentru pașii următori.

3.1 Curățare și standardizare date
3.1.1 Standardizarea câmpurilor de timp (TEXT)

Majoritatea tabelelor conțin câmpuri de tip timestamp (date_created, date_modified), stocate ca string-uri și, uneori, cu informații de timezone inconsistente. Pentru a permite calcule corecte de durată și ordonare temporală, toate aceste câmpuri sunt convertite la format datetime și standardizate la UTC.

In [44]:
import pandas as pd

def parse_datetime(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce", utc=True)
    return df

# aplicăm conversia
fires = parse_datetime(fires, ["date_created", "date_modified"])
fire_changelog = parse_datetime(fire_changelog, ["date_created"])
fire_perimeters = parse_datetime(fire_perimeters, ["date_created", "date_modified"])
fire_perimeters_changelog = parse_datetime(fire_perimeters_changelog, ["date_created"])

evac_zones = parse_datetime(evac_zones, ["date_created", "date_modified"])
evac_zone_changelog = parse_datetime(evac_zone_changelog, ["date_created"])
evac_zone_map = parse_datetime(evac_zone_map, ["date_created"])


3.1.2 Standardizarea identificatorilor (TEXT)

În urma Data Discovery, s-a observat că anumite identificatoare (geo_event_id, id) sunt stocate ca valori float, ca urmare a existenței unor valori lipsă. Pentru a preveni erori subtile în îmbinarea tabelelor, aceste câmpuri sunt convertite explicit la tipul Int64 (integer nullable).

In [45]:
def cast_int(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = df[c].astype("Int64")
    return df

fires = cast_int(fires, ["id"])
fire_changelog = cast_int(fire_changelog, ["id", "geo_event_id", "user_created_id"])
fire_perimeters = cast_int(fire_perimeters, ["id", "geo_event_id"])
fire_perimeters_changelog = cast_int(fire_perimeters_changelog, ["id", "fire_perimeter_id"])

evac_zones = cast_int(evac_zones, ["id", "region_id"])
evac_zone_changelog = cast_int(evac_zone_changelog, ["id", "evac_zone_id"])
evac_zone_map["geo_event_id"] = evac_zone_map["geo_event_id"].astype("Int64")


3.1.3 Curățarea datelor SVI (TEXT)

Setul de date SVI 2022 utilizează valoarea -999 pentru a indica lipsa unui scor de vulnerabilitate. Aceste valori trebuie tratate explicit ca valori lipsă pentru a evita distorsionarea analizelor statistice și a modelelor predictive.

In [46]:
# coloane percentile (RPL / EPL / EP)
svi_cols = [c for c in svi.columns if c.startswith(("RPL_", "EPL_", "EP_"))]

svi[svi_cols] = svi[svi_cols].replace(-999, pd.NA)

# verificare
svi[svi_cols].isna().mean().sort_values(ascending=False).head()


,0
RPL_THEME1,0.009249
RPL_THEMES,0.009249
RPL_THEME2,0.009058
EP_SNGPNT,0.009047
EP_HBURD,0.009047


3.2 Parsarea câmpurilor JSON
3.2.1 Parsarea fires.data (TEXT)

Câmpul data din tabelul geo_events_geoevent conține informații operaționale despre incendiu (ex. suprafață arsă, grad de containment, forward progress). Aceste informații sunt extrase în coloane separate pentru a putea fi utilizate în analiză și modelare.

In [47]:
import json

def extract_fire_data(val):
    if pd.isna(val):
        return {}
    try:
        return json.loads(val)
    except:
        return {}

fire_data = fires["data"].apply(extract_fire_data)

fires["acreage"] = fire_data.apply(lambda x: x.get("acreage"))
fires["containment"] = fire_data.apply(lambda x: x.get("containment"))
fires["is_fps"] = fire_data.apply(lambda x: x.get("is_fps"))


3.2.2 Parsarea fire_changelog.changes (TEXT)

Changelog-ul incendiilor conține rapoarte radio și modificări succesive ale stării incendiului. În această etapă, extragem semnalele timpurii de risc care pot indica necesitatea unei evacuări.

In [48]:
def extract_change(val, key):
    if pd.isna(val):
        return None
    try:
        d = json.loads(val)
        if key in d:
            return d[key][1]  # noua valoare
    except:
        return None
    return None

fire_changelog["rate_of_spread"] = fire_changelog["changes"].apply(
    lambda x: extract_change(x, "radio_traffic_indicates_rate_of_spread")
)

fire_changelog["structure_threat"] = fire_changelog["changes"].apply(
    lambda x: extract_change(x, "radio_traffic_indicates_structure_threat")
)

fire_changelog["spotting"] = fire_changelog["changes"].apply(
    lambda x: extract_change(x, "radio_traffic_indicates_spotting")
)


3.2.3 Parsarea evac_zone_changelog.changes (TEXT)

Pentru analiza întârzierilor, sunt relevante doar modificările de tip status (ex. advisory, warning, order). Alte modificări (ex. geometrie) sunt ignorate.

In [49]:
evac_zone_changelog["new_status"] = evac_zone_changelog["changes"].apply(
    lambda x: extract_change(x, "status")
)

# păstrăm doar rândurile cu schimbare de status
evac_zone_status_changes = evac_zone_changelog[
    evac_zone_changelog["new_status"].notna()
].copy()

evac_zone_status_changes["new_status"].value_counts()


,count
new_status,
warnings,1263
orders,659
advisories,538
shelter_in_place,7


Curățarea și procesarea inițială a datelor a confirmat că informația critică pentru analiza întârzierilor este concentrată în changelog-urile de incendii și de evacuare. După tratarea valorilor codificate special în datele SVI și filtrarea schimbărilor relevante de status, setul de date rezultat conține un număr suficient de evenimente de evacuare (warnings și orders) pentru a permite o analiză temporală robustă. Proporția redusă de valori lipsă în scorurile de vulnerabilitate indică o bună acoperire a datelor externe și susține analiza disparităților în etapele următoare.

## Pasul 3.2 — Definirea unității de analiză
3.2.0 De ce este critic acest pas?

Alegerea unității de analiză determină:

cum definim variabila țintă,

ce informație este disponibilă la momentul predicției,

dacă există sau nu data leakage,

ce tip de modele pot fi utilizate ulterior.

În contextul evacuărilor, datele sunt profund temporale: semnalele de risc apar progresiv, iar ordinele de evacuare sunt emise la momente discrete. Prin urmare, este necesar să alegem o unitate de analiză care să reflecte procesul decizional real și să permită evaluarea impactului anticipativ al unui model.

Varianta B — Fire–time (observații longitudinale)

mai multe observații per incendiu, fiecare corespunzând unui moment în timp

fiecare observație folosește doar informația disponibilă până la acel moment

Avantaje:

reflectă procesul real de monitorizare

permite predicții de tip “va fi nevoie de evacuare în următoarele X ore?”

permite simularea impactului (câte minute am câștigat)

Limitări:

implementare mai complexă

necesită control strict al temporalității

În acest proiect adoptăm Varianta B – fire–time, deoarece:

Track 1 vizează reducerea întârzierilor operaționale, nu doar analiza post-hoc;

un model anticipativ trebuie să funcționeze pe informație parțială, disponibilă în timp real;

această abordare permite evaluarea directă a impactului soluției propuse.

Cu toate acestea, vom construi agregări fire-level ca baseline și pentru interpretare, dar modelarea principală se va baza pe unitatea fire–time.

Pentru fiecare incendiu (geo_event_id), construim observații la intervale regulate de timp, de exemplu:

la fiecare 1 oră de la apariția primului semnal relevant,

până la momentul primei evacuări (sau până la finalul observației).

La fiecare moment t, observația include:

doar informația disponibilă până la t,

un target care indică dacă o evacuare va avea loc în următoarele X ore.

In [52]:
# start_time: din fires
start_time = fires.set_index("id")["date_created"]

# end_time: max între changelog și fires.date_modified
end_time_changelog = fire_changelog.groupby("geo_event_id")["date_created"].max()
end_time_fires = fires.set_index("id")["date_modified"]

end_time = pd.concat([end_time_changelog, end_time_fires], axis=1).max(axis=1)

timeline_bounds = pd.DataFrame({
    "start_time": start_time,
    "end_time": end_time
})

timeline_bounds.head()


,start_time,end_time
76,2021-08-11 00:09:56.481066+00:00,2023-02-09 20:34:24.180117+00:00
77,2021-08-11 07:21:46.054995+00:00,2023-02-09 20:34:24.225186+00:00
78,2021-08-11 21:02:16.301416+00:00,2023-02-09 20:34:24.266124+00:00
79,2021-08-12 01:46:31.043052+00:00,2023-02-09 20:34:24.308048+00:00
80,2021-08-12 02:40:30.939331+00:00,2023-02-09 20:34:24.358582+00:00


In [ ]:
def build_time_grid(start, end, freq="1H"):
    if pd.isna(start) or pd.isna(end) or start >= end:
        return []
    return pd.date_range(start=start, end=end, freq=freq)

records = []

for fire_id, row in timeline_bounds.iterrows():
    times = build_time_grid(row["start_time"], row["end_time"])
    for t in times:
        records.append({
            "geo_event_id": fire_id,
            "timestamp": t
        })

fire_time_df = pd.DataFrame(records)

fire_time_df.shape, fire_time_df.head()


/tmp/ipython-input-2158567069.py:4: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  return pd.date_range(start=start, end=end, freq=freq)
